# 06 · Errors & Logs

**Focus:** asserting that code raises the right exceptions and emits the right output.

Testing the *happy path* is only half the job. Robust code also fails loudly and correctly, and
good tests pin that behavior down:

- **`pytest.raises(...)`** — assert that a block raises a specific exception (and optionally that its
  message matches). If the block *doesn't* raise, the test fails.
- **`caplog`** — a built-in fixture that captures log records so you can assert on log level and text.
- **`capsys`** — a built-in fixture that captures `stdout`/`stderr` so you can assert on `print` output.

### Setup — point the shell at this course's tools
The `!` cells below run command-line tools (`pytest`, later `mutmut`, `playwright`). We prepend the
active kernel's `bin/` directory to `PATH` so those commands resolve to the versions installed for
**this course**, not some other Python on your machine. Run this cell first.

In [1]:
import sys, os
# The kernel's own interpreter lives in the course virtualenv's bin/ directory.
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]
print("Using Python:", sys.executable)
!pytest --version

Using Python: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python


pytest 9.1.1


### Code that raises and logs
A withdrawal function: it rejects bad input with a `ValueError` and logs a warning on overdraft attempts.

In [2]:
%%writefile nb06_bank.py
import logging

logger = logging.getLogger("bank")

def withdraw(balance: float, amount: float) -> float:
    if amount <= 0:
        raise ValueError("amount must be positive")
    if amount > balance:
        logger.warning("overdraft attempt: balance=%s amount=%s", balance, amount)
        raise ValueError("insufficient funds")
    print(f"withdrew {amount}, remaining {balance - amount}")
    return balance - amount

Writing nb06_bank.py


### Test the exceptions with `pytest.raises`
`match=` takes a **regex** checked against the exception message. This asserts both *that* it raised
and *why*.

In [3]:
%%writefile test_nb06.py
import logging
import pytest
from nb06_bank import withdraw

def test_rejects_negative_amount():
    with pytest.raises(ValueError, match="must be positive"):
        withdraw(100, -5)

def test_rejects_overdraft():
    with pytest.raises(ValueError, match="insufficient funds"):
        withdraw(100, 500)

def test_overdraft_is_logged(caplog):
    # caplog captures log records emitted during the block.
    with caplog.at_level(logging.WARNING):
        with pytest.raises(ValueError):
            withdraw(100, 500)
    assert "overdraft attempt" in caplog.text
    assert caplog.records[0].levelname == "WARNING"

def test_success_prints_receipt(capsys):
    remaining = withdraw(100, 30)
    assert remaining == 70
    captured = capsys.readouterr()          # grab everything printed so far
    assert "withdrew 30, remaining 70" in captured.out

Writing test_nb06.py


### Run it

In [4]:
!pytest -v test_nb06.py

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson
plugins: syrupy-5.5.3, mock-3.15.1, cov-7.1.0, xdist-3.8.0, asyncio-1.4.0, time-machine-3.2.0, anyio-4.14.2, base-url-2.1.0, playwright-0.8.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 4 items                                                              

test_nb06.py::test_rejects_negative_amount PASSED                        [ 25%]
test_nb06.py::test_rejects_overdraft PASSED                              [ 50%]
test_nb06.py::test_overdraft_is_logged 

PASSED                            [ 75%]
test_nb06.py::test_success_prints_receipt PASSED                         [100%]

============================== 4 passed in 0.03s ===============================


### ⚠️ Common pitfall — `match=` is a regex, and empty `raises` blocks hide bugs
`pytest.raises(ValueError, match="...")` treats the pattern as a **regular expression**, so characters
like `(`, `)`, `.`, `$` are special and may need escaping. Also, an empty `with pytest.raises(...):`
block that wraps *too much* code can pass for the wrong reason — keep the block down to the single call
you expect to raise.

### 🔬 Try it yourself
`pytest.raises` can *capture* the exception object for deeper assertions via `as exc_info`. Run this to
inspect the caught exception's type and message. Then **try** changing `"insufficient"` to text that
isn't in the message and re-run to see it fail.

In [5]:
%%writefile test_nb06_tryit.py
import pytest
from nb06_bank import withdraw

def test_capture_the_exception():
    with pytest.raises(ValueError) as exc_info:
        withdraw(100, 500)
    print("exception type   :", exc_info.type.__name__)
    print("exception message:", exc_info.value)
    assert "insufficient" in str(exc_info.value)   # <-- change this text to see the test fail

Writing test_nb06_tryit.py


In [6]:
!pytest -v -s test_nb06_tryit.py

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson
plugins: syrupy-5.5.3, mock-3.15.1, cov-7.1.0, xdist-3.8.0, asyncio-1.4.0, time-machine-3.2.0, anyio-4.14.2, base-url-2.1.0, playwright-0.8.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 1 item                                                               

test_nb06_tryit.py::test_capture_the_exception exception type   : ValueError
exception message: insufficient funds
PASSED

============================== 1 passed in 0.02s ===============================


### What you learned
- `with pytest.raises(ExcType, match="regex"):` asserts a block raises the expected error *and* message.
- `caplog` captures log records — assert on `caplog.text` and `caplog.records[i].levelname`.
- `capsys.readouterr()` returns captured `.out` / `.err` so you can assert on printed output.

Next up: **async testing**.